# UTS GRAF SAINS DATA 2023E

* Nama: Nabila Nasywa Zen
* NIM: 23031554115
* Kelas: Graf Sains Data 2023E

## Membaca File event.csv

In [2]:
import pandas as pd

data = pd.read_csv("data/events.csv")
data.head(5)

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


## Connection to Neo4j

In [3]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "Zenida2004"))
driver.verify_connectivity()

print("Berhasil terhubung ke Neo4j!")

Berhasil terhubung ke Neo4j!


/var/folders/44/z3ss0qj92jd0l22wd4xsj9br0000gn/T/ipykernel_44566/1694224761.py:4: ExperimentalWarning: The configuration may change in the future.
  driver.verify_connectivity()


In [4]:
with driver.session(database="neo4j") as session:
    result = session.run("RETURN 'Neo4j siap digunakan' AS status")
    print(result.single()["status"])

Neo4j siap digunakan


In [5]:
import os 
import sys

csv_file_path = os.path.join("data", "events.csv")

if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
    sys.exit(1)

print(f"File ditemukan: {csv_file_path}")

File ditemukan: data/events.csv


### Create Constraint

In [6]:
def create_constraints(session):
    session.run(
        """
        CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS
        FOR (v:Visitor)
        REQUIRE v.visitorid IS UNIQUE
        """
    ).consume()

    session.run(
        """
        CREATE CONSTRAINT item_id_unique IF NOT EXISTS
        FOR (i:Item)
        REQUIRE i.itemid IS UNIQUE
        """
    ).consume()

    print("Constraints berhasil diperiksa/dibuat.")

In [7]:
with driver.session(database="neo4j") as session:
    create_constraints(session)

Constraints berhasil diperiksa/dibuat.


digunakan untuk membuat aturan agar ID setiap node Visitor dan Item di Neo4j bersifat unik.

### Import Batch

In [ ]:
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorid: row.visitorid})
    MERGE (i:Item {itemid: row.itemid})
    WITH v, i, row

    CALL {
        WITH v, i, row
        WITH v, i, row
        WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row
        WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row
        WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {
            timestamp: row.timestamp,
            transactionid: row.transactionid
        }]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)


### Import Data

In [ ]:
import tqdm
import math

batch_size = 5000

with driver.session(database="neo4j") as session:
    create_constraints(session)

    chunks = pd.read_csv(csv_file_path, chunksize=batch_size)

    total_rows = sum(1 for _ in open(csv_file_path)) - 1
    total_chunks = math.ceil(total_rows / batch_size)
    print("Memulai proses impor data ke Neo4j...")

    for chunk in tqdm.tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
        chunk["transactionid"] = (chunk["transactionid"].fillna("").astype(str))
        records = chunk.to_dict(orient="records")
        session.write_transaction(import_batch, records)

driver.close()
print("Proses impor berhasil.")

Constraints berhasil diperiksa/dibuat.
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data: 100%|██████████| 552/552 [02:16<00:00,  4.04it/s]

Proses impor berhasil.


## Soal 1

#### Melihat tanggal paling awal dan paling akhir pada dataset

In [8]:
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "Zenida2004"))
driver.verify_connectivity()

with driver.session(database="neo4j") as session:
    result = session.run("""
        MATCH ()-[r]->()
        WHERE r.timestamp IS NOT NULL
        RETURN
            min(r.timestamp) AS min_timestamp,
            max(r.timestamp) AS max_timestamp,
            count(r) AS total_events
    """)
    data = result.single()

tanggal_awal = data["min_timestamp"]
tanggal_maksimal = data["max_timestamp"]

print("Total event:", data["total_events"])
print("Tanggal awal:", pd.to_datetime(tanggal_awal, unit="ms", utc=True))
print("Tanggal akhir:", pd.to_datetime(tanggal_maksimal, unit="ms", utc=True))

/var/folders/44/z3ss0qj92jd0l22wd4xsj9br0000gn/T/ipykernel_51288/2618214975.py:5: ExperimentalWarning: The configuration may change in the future.
  driver.verify_connectivity()


Total event: 8268307
Tanggal awal: 2015-05-03 03:00:04.384000+00:00
Tanggal akhir: 2025-06-15 15:06:42+00:00


Cypher query ini digunakan untuk mengetahui rentang waktu dan jumlah event yang ada di Neo4j. 'MATCH ()-[r]->()' digunakan untuk mengambil semua relasi yang ada di graf, kemudian 'WHERE r.timestamp IS NOT NULL' memastikan hanya relasi yang memiliki timestamp yang dihitung. Selanjutnya, 'min(r.timestamp)' digunakan untuk mencari timestamp paling awal, 'max(r.timestamp)' untuk mencari timestamp paling akhir, dan 'count(r)' untuk menghitung total relasi yang memiliki timestamp. Hasilnya digunakan untuk mengetahui cakupan waktu dan jumlah event yang tersimpan di dalam graf.

Terdapat total event 2.756.101 dengan tanggal awal 3 Mei 2015 dan tanggal akhir 18 September 2015.

## Soal 2

#### Memfilter data interaksi berdasarkan range tanggal yang telah ditetapkan dan simpan ke CSV.

In [9]:
import pandas as pd
import os

input_file = "data/events.csv"
output_dir = "data/filtered"
os.makedirs(output_dir, exist_ok=True)

p1_file = os.path.join(output_dir, "events_P1_training.csv")
p2_file = os.path.join(output_dir, "events_P2_testing.csv")

p1_awal = pd.Timestamp("2015-05-15", tz="UTC")
p1_akhir = pd.Timestamp("2015-08-04", tz="UTC")

p2_awal = pd.Timestamp("2015-08-04", tz="UTC")
p2_akhir = pd.Timestamp("2015-08-25", tz="UTC")

total_p1 = 0
total_p2 = 0

for chunk in pd.read_csv(input_file, chunksize=200_000):
    event_dates = pd.to_datetime(chunk["timestamp"], unit="ms", utc=True)

    mask_p1 = (event_dates >= p1_awal) & (event_dates < p1_akhir)
    mask_p2 = (event_dates >= p2_awal) & (event_dates < p2_akhir)

    if mask_p1.any():
        chunk.loc[mask_p1].to_csv(p1_file, mode="a", index=False, header=not os.path.exists(p1_file))
        total_p1 += mask_p1.sum()

    if mask_p2.any():
        chunk.loc[mask_p2].to_csv(p2_file, mode="a", index=False, header=not os.path.exists(p2_file))
        total_p2 += mask_p2.sum()

print("Jumlah event P1 training:", total_p1)
print("Jumlah event P2 testing:", total_p2)


Jumlah event P1 training: 1717044
Jumlah event P2 testing: 373717


Setelah menjalani proses fold untuk Data Training dan Testing, dengan Data Training yang dipilih adalah 15 Mei - 3 Agustus 2015 terdapat 1.717.044 jumlah interaksi. Serta pada Data Testing pada tanggal 4-24 Agustus 2015 terdapat 373.717 jumlah interaksi.

## Soal 3

In [10]:
def komposisi_graph(session, data_awal, data_akhir):
    query = """
    MATCH (v:Visitor)-[r]->(i:Item)
    WHERE r.timestamp >= $start_ts
      AND r.timestamp < $end_ts
    RETURN
        type(r) AS relationship_type,
        count(r) AS total_relationships,
        count(DISTINCT v) AS total_visitors,
        count(DISTINCT i) AS total_items
    ORDER BY relationship_type
    """

    hasil = session.run(
        query,
        start_ts=int(pd.Timestamp(data_awal, tz="UTC").timestamp() * 1000),
        end_ts=int(pd.Timestamp(data_akhir, tz="UTC").timestamp() * 1000)
    )

    return pd.DataFrame([record.data() for record in hasil])


with driver.session(database="neo4j") as session:
    p1_composition = komposisi_graph(session, "2015-05-15", "2015-08-04")
    p2_composition = komposisi_graph(session, "2015-08-04", "2015-08-25")

print("=== Komposisi Subgraf P1 (Training) ===")
display(p1_composition)

print("=== Komposisi Subgraf P2 (Testing) ===")
display(p2_composition)

=== Komposisi Subgraf P1 (Training) ===


,relationship_type,total_relationships,total_visitors,total_items
0,ADDED_TO_CART,126399,23116,17035
1,BOUGHT,41754,7270,8267
2,VIEWED,4982979,877402,191506


=== Komposisi Subgraf P2 (Testing) ===


,relationship_type,total_relationships,total_visitors,total_items
0,ADDED_TO_CART,29943,5555,5775
1,BOUGHT,9360,1682,2375
2,VIEWED,1081848,212666,92260


Berdasarkan komposisi subgraf diatas, diperoleh 1.717.044 jumlah interaksi pada periode training (P1), sedangkan pada periode testing (P2) didapatkan 373.717 jumlah interaksi. Pada kedua periode, terlihat pada relasi 'VIEWED' merupakan jenis interaksi paling dominan jika dibandingkan dengan yang lain, diikuti oleh 'ADDED_TO_CART' dan 'BOUGHT'. Hal ini berarti aktivitas user lebih sering hanya melihat-lihat produk jika dibandingkan dengan kegiatan menambhakn produk ke dalam keranjang maupun melakukan pembelian. 

In [11]:
def get_overall_composition(session, start_date, end_date):
    start_ts = int(pd.Timestamp(start_date, tz="UTC").timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_date, tz="UTC").timestamp() * 1000)

    query = """
    MATCH (v:Visitor)-[r]->(i:Item)
    WHERE r.timestamp >= $start_ts
      AND r.timestamp < $end_ts
    RETURN
        count(r) AS total_interactions,
        count(DISTINCT v) AS unique_visitors,
        count(DISTINCT i) AS unique_items
    """

    record = session.run(query, start_ts=start_ts, end_ts=end_ts).single()
    return record.data()

with driver.session(database="neo4j") as session:
    overall_p1 = get_overall_composition(session, "2015-05-15", "2015-08-04")
    overall_p2 = get_overall_composition(session, "2015-08-04", "2015-08-25")

composition_summary = pd.DataFrame([{"Periode": "P1 - Training", **overall_p1}, {"Periode": "P2 - Testing", **overall_p2}])
composition_summary

,Periode,total_interactions,unique_visitors,unique_items
0,P1 - Training,5151132,879318,191682
1,P2 - Testing,1121151,213262,92459


Cypher query ini digunakan untuk menghitung komposisi graf berdasarkan rentang waktu yang sudah ditentukan. 'MATCH' digunakan untuk mengambil hubungan antara node Visitor dan Item, sedangkan 'WHERE' menyaring interaksi berdasarkan timestamp awal dan akhir. Selanjutnya, 'count(r)' menghitung jumlah seluruh interaksi, 'count(DISTINCT v)' menghitung jumlah Visitor unik, dan 'count(DISTINCT i)' menghitung jumlah Item unik. Hasilnya digunakan untuk melihat jumlah interaksi, Visitor, dan Item pada masing-masing periode training dan testing.

Berdasarkan hasil query komposisi subgraf, periode training (P1) terdiri atas 1.717.044 interaksi, 879.318 node Visitor unik, dan 191.682 node Item unik. Sementara itu, periode testing (P2) terdiri atas 373.717 interaksi, 213.262 node Visitor unik, dan 92.459 node Item unik.

## Soal 4 

In [12]:
import numpy as np
from scipy.sparse import coo_matrix

events_p1 = pd.read_csv("data/filtered/events_P1_training.csv")

kode_visitor, visitor_id = pd.factorize(events_p1["visitorid"])
kode_item, item_id = pd.factorize(events_p1["itemid"])

user_item_matrix = coo_matrix(
    (np.ones(len(events_p1), dtype=np.int8), (kode_visitor, kode_item)),
    shape=(len(visitor_id), len(item_id))
).tocsr()

user_item_matrix.data[:] = 1
user_item_matrix.eliminate_zeros()

print("Jumlah baris data P1:", len(events_p1))
print("Jumlah Visitor unik:", len(visitor_id))
print("Jumlah Item unik:", len(item_id))
print("Dimensi matriks VSM User:", user_item_matrix.shape)
print("Jumlah pasangan Visitor-Item unik:", user_item_matrix.nnz)
print("Tipe matriks:", type(user_item_matrix))

Jumlah baris data P1: 5151132
Jumlah Visitor unik: 879318
Jumlah Item unik: 191682
Dimensi matriks VSM User: (879318, 191682)
Jumlah pasangan Visitor-Item unik: 1334999
Tipe matriks: <class 'scipy.sparse._csr.csr_matrix'>


VSM User dibentuk dari 3.434.088 baris data training P1, yang terdiri dari 879.318 Visitor unik dan 191.682 Item unik. Matriks yang dihasilkan memiliki shape '(879318, 191682)', dengan baris merepresentasikan Visitor dan kolom merepresentasikan Item. Terdapat 1.334.999 pasangan Visitor–Item unik yang memiliki interaksi. Matriks disimpan dalam format 'csr_matrix' (*Compressed Sparse Row*) untuk menghemat memori karena sebagian besar pasangan Visitor–Item tidak memiliki interaksi.

Jumlah pasangan Visitor–Item unik lebih kecil daripada jumlah event karena seorang Visitor dapat berinteraksi dengan Item yang sama berkali-kali. Dalam matriks biner ini, interaksi berulang tersebut direpresentasikan sebagai satu nilai 1.

## Soal 5

In [13]:
item_user_matrix = user_item_matrix.transpose().tocsr()

print("Dimensi matriks VSM Item:", item_user_matrix.shape)
print("Jumlah Item unik:", item_user_matrix.shape[0])
print("Jumlah Visitor unik:", item_user_matrix.shape[1])
print("Jumlah pasangan Item-Visitor unik:", item_user_matrix.nnz)
print("Tipe matriks:", type(item_user_matrix))


Dimensi matriks VSM Item: (191682, 879318)
Jumlah Item unik: 191682
Jumlah Visitor unik: 879318
Jumlah pasangan Item-Visitor unik: 1334999
Tipe matriks: <class 'scipy.sparse._csr.csr_matrix'>


VSM Item dibentuk dengan mentraspose matriks VSM User, sehingga baris merepresentasikan Item dan kolom merepresentasikan Visitor. Berdasarkan data training P1, terdapat 191.682 Item unik dan 879.318 Visitor unik, dengan dimensi matriks sebesar '(191682, 879318)'. Jumlah pasangan Item–Visitor yang memiliki interaksi adalah 1.334.999. Matriks disimpan dalam format 'csr_matrix' agar penggunaan memori lebih efisien karena sebagian besar pasangan Item dan Visitor tidak memiliki interaksi. Matriks ini dapat digunakan untuk melihat pola interaksi setiap Item berdasarkan Visitor yang pernah mengaksesnya.

## Soal 6 

Berdasarkan analisis VSM User dan VSM Item, pada proyeksi Item lebih berpotensi untuk lebih efisien dalam dilakukannya analisis kemiripan karena jumlah Item unik sebanyak 191.682, jauh lebih sedikit jika dibandingkan dengan 879.318 pada User unik. Hal ini dikarenakan jumlah simpul yang lebih sedikit dapat mengurangi beban komputasi saat membentuk proyeksi graf.

## Soal 7

In [16]:
import numpy as np
from scipy.sparse import coo_matrix
import pandas as pd

events_p1 = pd.read_csv("data/filtered/events_P1_training.csv")

item_user_matrix = coo_matrix(
    (np.ones(len(events_p1), dtype=np.int8), (kode_item, kode_visitor)),
    shape=(len(item_id), len(visitor_id))
).tocsr()

item_user_matrix.data[:] = 1

item_norms = np.sqrt(np.asarray(item_user_matrix.multiply(item_user_matrix).sum(axis=1)).ravel())

reference_idx = int(np.argmax(item_norms))
reference_item_id = int(item_id[reference_idx])

dot_products = (item_user_matrix.getrow(reference_idx) @ item_user_matrix.T).toarray().ravel()

denominator = item_norms[reference_idx] * item_norms
similarities = np.divide(
    dot_products,
    denominator,
    out=np.zeros_like(dot_products, dtype=float),
    where=denominator != 0
)

similarities[reference_idx] = -1
top5_indices = np.argsort(similarities)[-5:][::-1]

top5_similar_items = pd.DataFrame({
    "reference_item_id": reference_item_id,
    "similar_item_id": [int(item_id[i]) for i in top5_indices],
    "cosine_similarity": [float(similarities[i]) for i in top5_indices]
})

print("Item acuan:", reference_item_id)
display(top5_similar_items)

Item acuan: 5411


,reference_item_id,similar_item_id,cosine_similarity
0,5411,33030,0.144373
1,5411,33956,0.084532
2,5411,358234,0.081491
3,5411,375492,0.065957
4,5411,459936,0.062732


Item 33030 memiliki nilai cosine similarity tertinggi terhadap Item 5411, yaitu 0,144373. Artinya, di antara lima Item tersebut, Item 33030 memiliki pola interaksi Visitor yang paling mirip dengan Item acuan. Nilai similarity ini didasarkan pada pola Visitor yang sama-sama berinteraksi dengan Item, bukan pada kemiripan nama atau kategori produknya.

## (a) 

In [17]:
similarity_records = top5_similar_items.to_dict(orient="records")
save_similarity_query = """
UNWIND $rows AS row
MATCH (source:Item {itemid: row.reference_item_id})
MATCH (target:Item {itemid: row.similar_item_id})
MERGE (source)-[r:SIMILAR_TO]->(target)
SET r.similarity = row.cosine_similarity
RETURN
    source.itemid AS reference_item,
    target.itemid AS similar_item,
    r.similarity AS cosine_similarity
ORDER BY cosine_similarity DESC
"""

with driver.session(database="neo4j") as session:
    saved_results = session.run(
        save_similarity_query,
        rows=similarity_records
    )
    saved_similarity = pd.DataFrame(
        [record.data() for record in saved_results]
    )

display(saved_similarity)

,reference_item,similar_item,cosine_similarity
0,5411,33030,0.144373
1,5411,33956,0.084532
2,5411,358234,0.081491
3,5411,375492,0.065957
4,5411,459936,0.062732


In [18]:
top5_query = """
MATCH (source:Item {itemid: 5411})-[r:SIMILAR_TO]->(target:Item)
RETURN
    source.itemid AS reference_item,
    target.itemid AS similar_item,
    r.similarity AS cosine_similarity
ORDER BY cosine_similarity DESC
LIMIT 5
"""

with driver.session(database="neo4j") as session:
    result = session.run(top5_query)
    top5_from_neo4j = pd.DataFrame(
        [record.data() for record in result]
    )

display(top5_from_neo4j)

,reference_item,similar_item,cosine_similarity
0,5411,33030,0.144373
1,5411,33956,0.084532
2,5411,358234,0.081491
3,5411,375492,0.065957
4,5411,459936,0.062732


# 2.2 Analytics

## Soal 1

In [ ]:
import pandas as pd
import numpy as np

events_p1 = pd.read_csv("data/filtered/events_P1_training.csv")
events_p2 = pd.read_csv("data/filtered/events_P2_testing.csv")

users_p1 = set(events_p1["visitorid"].unique())
users_p2 = set(events_p2["visitorid"].unique())

items_per_user = events_p1.groupby("visitorid")["itemid"].nunique()
active_users = set(items_per_user[items_per_user >= 5].index)

eligible_users = sorted(active_users.intersection(users_p1).intersection(users_p2))

random_seed = 42
rng = np.random.RandomState(random_seed)
selected_user = int(rng.choice(eligible_users))

print("Jumlah user P1:", len(users_p1))
print("Jumlah user P2:", len(users_p2))
print("Jumlah kandidat yang memenuhi syarat:", len(eligible_users))
print("User terpilih:", selected_user)
print("Jumlah item unik pada training:",
      int(items_per_user.loc[selected_user]))


Random seed: 42
Jumlah user P1: 879318
Jumlah user P2: 213262
Jumlah kandidat yang memenuhi syarat: 2225
User terpilih: 547162
Jumlah item unik pada training: 5


Pemilihan user dilakukan secara acak menggunakan random seed 42 agar proses pemilihan dapat direproduksi. Dari 879.318 user pada data training P1 dan 213.262 user pada data testing P2, diperoleh 2.225 kandidat yang memenuhi kriteria pemilihan. User yang terpilih adalah 547162, dengan 5 item unik pada data training yang digunakan untuk proses rekomendasi KNN.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

user_to_idx = {user_id: idx for idx, user_id in enumerate(visitor_id)}

target_idx = user_to_idx[selected_user]
target_items = set(user_item_matrix[target_idx].indices)

knn = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute")
knn.fit(user_item_matrix)

distances, indices = knn.kneighbors(user_item_matrix[target_idx])

scores = {}

for neighbor_idx, distance in zip(indices[0], distances[0]):
    if neighbor_idx == target_idx:
        continue

    similarity = 1 - distance

    for item_idx in user_item_matrix[neighbor_idx].indices:
        if item_idx not in target_items:
            scores[item_idx] = scores.get(item_idx, 0) + similarity

ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

top5_recommendations = pd.DataFrame({
    "itemid": [int(item_id[i]) for i, _ in ranked_items[:5]],
    "recommendation_score": [score for _, score in ranked_items[:5]]
})

print("User target:", selected_user)
print("Jumlah tetangga:", min(5, len(indices[0]) - 1))
print("Jumlah kandidat item baru:", len(scores))
display(top5_recommendations)


User target: 547162
Jumlah tetangga: 5
Jumlah kandidat item baru: 1


,itemid,recommendation_score
0,127042,0.516398


Berdasarkan hasil rekomendasi KNN dengan \(k=5\), user 547162 memperoleh satu rekomendasi item baru, yaitu Item 127042, dengan recommendation score sebesar 0,516398. Hanya satu item yang memenuhi kriteria rekomendasi karena item kandidat lainnya telah berinteraksi dengan user target atau tidak ditemukan dari lima tetangga terdekat.

In [27]:
print("Jumlah item milik user target:", len(target_items))

for neighbor_idx, distance in zip(indices[0], distances[0]):
    if neighbor_idx == target_idx:
        continue

    neighbor_items = set(user_item_matrix[neighbor_idx].indices)
    new_items = neighbor_items - target_items

    print(
        f"User tetangga: {visitor_id[neighbor_idx]} | "
        f"Total item: {len(neighbor_items)} | "
        f"Item baru: {len(new_items)} | "
        f"Similarity: {1 - distance:.4f}"
    )

Jumlah item milik user target: 5
User tetangga: 845079 | Total item: 2 | Item baru: 0 | Similarity: 0.6325
User tetangga: 647220 | Total item: 2 | Item baru: 0 | Similarity: 0.6325
User tetangga: 1049493 | Total item: 2 | Item baru: 0 | Similarity: 0.6325
User tetangga: 835934 | Total item: 2 | Item baru: 0 | Similarity: 0.6325
User tetangga: 969217 | Total item: 3 | Item baru: 1 | Similarity: 0.5164


Berdasarkan hasil pemeriksaan, user target memiliki 5 item pada data training. Dari lima tetangga terdekat, empat user masing-masing memiliki 2 item dengan nilai similarity 0,6325, tetapi tidak memiliki item baru yang dapat direkomendasikan. Sementara itu, user 969217 memiliki 3 item dengan nilai similarity 0,5164 dan menyumbangkan 1 item baru yang belum pernah diakses oleh user target. Oleh karena itu, model KNN hanya menghasilkan satu rekomendasi, yaitu Item 127042, dengan recommendation score sebesar 0,516398.

## Soal 2

In [26]:
actual_items = set(events_p2.loc[events_p2["visitorid"] == selected_user, "itemid" ].unique())
recommended_items = top5_recommendations["itemid"].tolist()

hits = [1 if item in actual_items else 0 for item in recommended_items]
hit_at_5 = int(sum(hits) > 0)
dcg_at_5 = sum(rel / np.log2(rank + 1)for rank, rel in enumerate(hits, start=1))

ideal_hits = min(len(actual_items), 5)
idcg_at_5 = sum(
    1 / np.log2(rank + 1)
    for rank in range(1, ideal_hits + 1)
)

ndcg_at_5 = dcg_at_5 / idcg_at_5 if idcg_at_5 > 0 else 0.0

print("User target:", selected_user)
print("Jumlah item aktual di P2:", len(actual_items))
print("Item aktual:", sorted(actual_items))
print("Item rekomendasi:", recommended_items)
print("Relevansi tiap item:", hits)
print("Hit@5:", hit_at_5)
print("NDCG@5:", round(ndcg_at_5, 4))


User target: 547162
Jumlah item aktual di P2: 4
Item aktual: [np.int64(123554), np.int64(346300), np.int64(363290), np.int64(413424)]
Item rekomendasi: [127042]
Relevansi tiap item: [0]
Hit@5: 0
NDCG@5: 0.0


Berdasarkan hasil evaluasi, nilai Hit@5 yang diperoleh adalah 0 dan NDCG@5 sebesar 0,0. Hal ini terjadi karena item yang direkomendasikan oleh model, yaitu Item 127042, tidak termasuk dalam empat item yang benar-benar diakses oleh user pada data testing P2. Jadi, meskipun model berhasil memberikan rekomendasi, item tersebut belum sesuai dengan interaksi aktual user. Akibatnya, tidak ada item relevan yang ditemukan dan kedua metrik evaluasi menghasilkan nilai 0.